# Fundamentos de Calidad de Datos (Junior)

Objetivos:
- Entender las dimensiones de calidad de datos.
- Medir y evaluar la calidad de un dataset.
- Identificar problemas comunes y estrategias de corrección.

## 1. ¿Qué es Calidad de Datos?

Calidad = grado en que los datos cumplen los requisitos para su uso previsto.

**Dimensiones clave** (DAMA-DMBOK):
- **Precisión (Accuracy):** Datos reflejan la realidad.
- **Completitud (Completeness):** Sin nulos inesperados.
- **Consistencia (Consistency):** Coherencia entre sistemas/fuentes.
- **Validez (Validity):** Cumplen reglas de negocio (formato, rango).
- **Unicidad (Uniqueness):** Sin duplicados no deseados.
- **Puntualidad (Timeliness):** Actualizados a tiempo.

## 2. Ejemplo: Dataset con Problemas de Calidad

In [ ]:
import pandas as pd
import numpy as np

# Dataset simulado con problemas
data = {
    'cliente_id': [1, 2, None, 4, 5, 5],
    'nombre': ['Ana', 'Luis', 'Carlos', 'Marta', 'Pedro', 'Pedro'],
    'email': ['ana@example.com', 'invalido', 'carlos@x.com', None, 'pedro@x.com', 'pedro@x.com'],
    'edad': [28, 200, 35, -5, 42, 42],
    'saldo': [1500.0, 2300.5, None, 800.0, 3200.0, 3200.0],
    'pais': ['CO', 'CO', 'PE', 'MX', 'AR', 'AR']
}
df = pd.DataFrame(data)
print(df)

## 3. Medición de Calidad por Dimensión

In [ ]:
# 1. Completitud: % de valores no nulos
completitud = (1 - df.isna().mean()) * 100
print('Completitud (%):\n', completitud.round(1))

# 2. Unicidad: identificar duplicados
duplicados = df.duplicated(subset=['cliente_id'], keep=False).sum()
print('\nFilas con cliente_id duplicado:', duplicados)

# 3. Validez: edad en rango razonable [0, 120]
edad_valida = df['edad'].between(0, 120).sum()
print('Edades válidas:', edad_valida, 'de', df['edad'].count())

# 4. Formato email: regex básico
import re
patron_email = r'^[\w\.-]+@[\w\.-]+\.\w+$'
df['email_valido'] = df['email'].apply(
    lambda x: bool(re.match(patron_email, str(x))) if pd.notna(x) else False
)
print('\nEmails válidos:', df['email_valido'].sum(), 'de', df['email'].notna().sum())

## 4. Reglas de Calidad como Assertions

In [ ]:
# Definir reglas y ejecutar
reglas = {
    'cliente_id_no_nulo': lambda d: d['cliente_id'].notna().all(),
    'edad_en_rango': lambda d: d['edad'].between(0, 120).all(),
    'sin_duplicados_cliente': lambda d: not d.duplicated(subset=['cliente_id'], keep=False).any(),
    'email_valido_si_presente': lambda d: (d['email'].notna() == d['email_valido']).all()
}

resultados = {nombre: regla(df) for nombre, regla in reglas.items()}
print('Resultados de calidad:')
for r, ok in resultados.items():
    print(f'  {r}: {"✓ OK" if ok else "✗ FALLO"}')

if not all(resultados.values()):
    print('\n⚠️ Dataset NO cumple con todas las reglas de calidad.')

## 5. Estrategias de Corrección

| Problema | Estrategia |
|----------|------------|
| Nulos | Imputar con media/moda/default o eliminar filas |
| Duplicados | Deduplicar con `drop_duplicates()` o lógica de merge |
| Valores fuera de rango | Filtrar o reemplazar (cap/floor) |
| Formato inválido | Validar en ingesta, rechazar o sanitizar |

In [ ]:
# Ejemplo de corrección
df_limpio = df.copy()
# Eliminar filas con cliente_id nulo
df_limpio = df_limpio[df_limpio['cliente_id'].notna()]
# Deduplicar por cliente_id (mantener primera)
df_limpio = df_limpio.drop_duplicates(subset=['cliente_id'], keep='first')
# Cap edad a [0, 120]
df_limpio['edad'] = df_limpio['edad'].clip(0, 120)
# Llenar saldo nulo con 0
df_limpio['saldo'].fillna(0, inplace=True)

print('Dataset limpio:\n', df_limpio)
print('\nFilas originales:', len(df), '| Después de limpieza:', len(df_limpio))

## 6. Ejercicio

Dado el dataset `ventas_raw` (inventado o cargar de CSV):
1. Calcula el % de completitud por columna.
2. Identifica duplicados por `pedido_id`.
3. Valida que `monto` sea positivo.
4. Reporta un resumen de calidad con métricas clave.

In [ ]:
# Ejercicio Práctico Mejorado: Análisis de Calidad de Ventas

# Dataset de ventas simulado con problemas realistas
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

# Generar dataset de ventas con problemas comunes
n_records = 100

ventas_data = {
    'pedido_id': list(range(1, n_records+1)) + [50, 75],  # Duplicados
    'fecha': [datetime(2024, 11, 1) + timedelta(days=x) for x in range(n_records)] + [None, None],
    'cliente_id': np.random.randint(1000, 2000, n_records + 2),
    'producto': np.random.choice(['Laptop', 'Mouse', 'Teclado', None, 'Monitor'], n_records + 2),
    'cantidad': np.random.randint(-2, 20, n_records + 2),  # Algunas negativas (error)
    'precio_unitario': np.random.uniform(10, 2000, n_records + 2),
    'monto_total': np.random.uniform(-50, 50000, n_records + 2),  # Algunos negativos
    'estado': np.random.choice(['Completado', 'Pendiente', 'Cancelado', 'INVALIDO'], n_records + 2)
}

ventas_raw = pd.DataFrame(ventas_data)

print("Dataset de Ventas (primeras 10 filas):")
print(ventas_raw.head(10))
print(f"\nTotal registros: {len(ventas_raw)}")

# TU TAREA:
# Usar el DataQualityFramework creado anteriormente o Pandas puro para:

# 1. Medir completitud por columna
print("\n" + "="*60)
print("1. COMPLETITUD POR COLUMNA")
print("="*60)
# completitud = ...
# print(completitud)

# 2. Identificar duplicados en pedido_id
print("\n" + "="*60)
print("2. ANÁLISIS DE DUPLICADOS")
print("="*60)
# duplicados = ...
# print(f"Pedidos duplicados: {duplicados}")

# 3. Validar que monto_total sea positivo y cantidad > 0
print("\n" + "="*60)
print("3. VALIDACIÓN DE RANGOS")
print("="*60)
# monto_invalido = ...
# cantidad_invalida = ...

# 4. Validar consistencia: monto_total ≈ cantidad * precio_unitario (±5%)
print("\n" + "="*60)
print("4. CONSISTENCIA DE CÁLCULOS")
print("="*60)
# ventas_raw['monto_calculado'] = ...
# inconsistencias = ...

# 5. Validar que estado esté en valores permitidos
print("\n" + "="*60)
print("5. VALORES PERMITIDOS")
print("="*60)
# estados_validos = ['Completado', 'Pendiente', 'Cancelado']
# estados_invalidos = ...

# 6. Generar reporte de calidad consolidado
print("\n" + "="*60)
print("6. REPORTE DE CALIDAD CONSOLIDADO")
print("="*60)
# reporte = {
#     'total_registros': len(ventas_raw),
#     'completitud_promedio': ...,
#     'duplicados': ...,
#     'montos_invalidos': ...,
#     'cantidades_invalidas': ...,
#     'inconsistencias_calculo': ...,
#     'estados_invalidos': ...
# }
# print(reporte)

print("\n💡 Consejo: Implementa las 6 tareas anteriores")
print("💡 Bonus: Usa el DataQualityFramework para automatizar las validaciones")

## 11. Mejores Prácticas y Recomendaciones

### ✅ DO (Haz esto)

1. **Shift Left:** Valida calidad lo más temprano posible en el pipeline
2. **Automatiza:** Integra checks en CI/CD, no validación manual
3. **Documenta:** Mantén catálogo de reglas de calidad con justificación
4. **Mide:** Establece SLOs de calidad (ej: completitud > 95%)
5. **Alerta:** Notifica a owners cuando la calidad degrada
6. **Versiona:** Guarda reglas de calidad en Git junto con código
7. **Contextualiza:** No todos los nulos son malos, depende del caso de uso

### ❌ DON'T (Evita esto)

1. **Validación manual:** Procesos ad-hoc sin automatización
2. **Ignorar warnings:** Warnings se convierten en errors después
3. **Sobrecorrección:** Imputar datos sin justificación crea datos falsos
4. **Sin ownership:** Cada dataset debe tener un responsable
5. **Validación solo al final:** Detectar errores tarde es costoso
6. **Checks sin contexto:** Reglas rígidas sin considerar el negocio

### 🎯 KPIs de Calidad Recomendados

| Métrica | Target | Frecuencia |
|---------|--------|------------|
| Completitud | > 95% | Diaria |
| Uniqueness (PKs) | 100% | Por carga |
| Freshness | < 4 horas | Continua |
| Schema stability | Sin cambios no comunicados | Por despliegue |
| Anomaly rate | < 1% | Diaria |
| Data quality score | > 90% | Semanal |

### 📚 Recursos para Profundizar

**Herramientas:**
- Great Expectations: https://greatexpectations.io/
- Soda: https://www.soda.io/
- dbt: https://docs.getdbt.com/docs/build/tests
- OpenLineage: https://openlineage.io/

**Lectura:**
- DAMA-DMBOK Chapter 13: Data Quality
- "Data Quality: The Accuracy Dimension" - Jack E. Olson
- "The Data Warehouse Toolkit" - Kimball (capítulos de calidad)

## 12. Resumen y Próximos Pasos

### Puntos Clave

✅ **Calidad de datos es multidimensional:** Precisión, completitud, consistencia, validez, unicidad, puntualidad

✅ **Medir es el primer paso:** Métricas cuantitativas dan visibilidad

✅ **Automatización es clave:** Herramientas modernas (Great Expectations, Soda, dbt tests)

✅ **Quality Gates en CI/CD:** Valida antes de promover entre ambientes

✅ **Observabilidad:** Monitorea calidad continuamente con dashboards y alertas

✅ **Ownership:** Cada dataset debe tener un responsable de calidad

### Checklist de Autoevaluación

Has completado este módulo si puedes:
- [ ] Explicar las 6 dimensiones de calidad de datos
- [ ] Calcular métricas de calidad en un dataset
- [ ] Implementar reglas de calidad como assertions
- [ ] Diseñar estrategias de corrección apropiadas
- [ ] Comparar herramientas modernas de calidad
- [ ] Proponer quality gates para un pipeline

### Próximos Niveles

**Nivel Mid:**
- Calidad en arquitecturas distribuidas (Data Mesh)
- Backfill y reprocesos con garantías de calidad
- Integración de calidad con catálogos y linaje
- Observabilidad avanzada y detección de anomalías

**Nivel Senior:**
- Governance de calidad a escala organizacional
- SLOs y error budgets para datos
- Cultura de calidad y DataOps
- Cumplimiento regulatorio (GDPR, BCBS 239)

## 8. Herramientas Modernas de Calidad de Datos (2025)

### 8.1 Frameworks de Testing de Datos

#### Great Expectations
Framework Python para validación declarativa de datos.

**Características:**
- Expectations (expectativas) como assertions reutilizables
- Validación en batch y streaming
- Documentación automática de calidad
- Integración con Airflow, dbt, Spark

**Ejemplo conceptual:**
```python
# Sintaxis Great Expectations
import great_expectations as gx

# Crear expectation suite
suite = gx.ExpectationSuite(name="clientes_suite")

# Definir expectativas
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="cliente_id")
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="edad", min_value=0, max_value=120
    )
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToMatchRegex(
        column="email", regex=r"^[\w\.-]+@[\w\.-]+\.\w+$"
    )
)
```

#### Soda Core
Plataforma de calidad de datos con SQL-like checks.

**Características:**
- Checks en YAML (legibles por negocio)
- Detección de anomalías
- Alertas a Slack/PagerDuty
- Soporta 20+ data warehouses

**Ejemplo conceptual:**
```yaml
# checks.yml
checks for clientes:
  - row_count > 0
  - missing_count(cliente_id) = 0
  - invalid_count(email) = 0:
      valid format: email
  - duplicate_count(cliente_id) = 0
  - avg(edad) between 18 and 65
  - freshness(updated_at) < 1d
```

#### dbt Tests
Tests integrados en transformaciones dbt.

**Tests genéricos:**
- `not_null`
- `unique`
- `accepted_values`
- `relationships` (foreign keys)

**Ejemplo:**
```yaml
# schema.yml
models:
  - name: dim_clientes
    columns:
      - name: cliente_id
        tests:
          - not_null
          - unique
      - name: edad
        tests:
          - dbt_utils.accepted_range:
              min_value: 0
              max_value: 120
```

### 8.2 Comparativa de Herramientas

| Característica | Great Expectations | Soda Core | dbt Tests | Pandas Manual |
|----------------|-------------------|-----------|-----------|---------------|
| **Curva aprendizaje** | Media | Baja | Baja | Muy baja |
| **Expresividad** | Alta (Python) | Alta (YAML) | Media | Alta (código) |
| **Integración CI/CD** | ✅ | ✅ | ✅ | ⚠️ Manual |
| **Documentación auto** | ✅ | ✅ | ✅ | ❌ |
| **Alertas** | ✅ | ✅ | ⚠️ Limitado | ❌ |
| **Detección anomalías** | ⚠️ Básico | ✅ | ❌ | ❌ |
| **Costo** | Open source | Freemium | Open source | Gratis |
| **Mejor para** | Pipelines complejos | Monitoreo continuo | Transformaciones | Exploración |

### 8.3 Observabilidad de Calidad: Monte Carlo, Datafold

**Monte Carlo Data:**
- Machine learning para detección de anomalías
- Alertas automáticas de schema drift
- Linaje de impacto cuando hay incidentes

**Datafold:**
- Data diff automático entre ambientes (dev vs prod)
- Validación de migraciones
- Testing de PRs con impacto en datos

In [ ]:
# Simulación de framework de calidad moderno
from dataclasses import dataclass
from typing import List, Dict, Any, Callable
from datetime import datetime
import pandas as pd

@dataclass
class QualityCheck:
    """Representa una verificación de calidad"""
    name: str
    column: str
    check_type: str  # 'not_null', 'unique', 'range', 'regex', 'custom'
    params: Dict[str, Any]
    severity: str = 'error'  # 'error', 'warning'
    
@dataclass
class QualityResult:
    """Resultado de una verificación"""
    check_name: str
    passed: bool
    rows_evaluated: int
    rows_failed: int
    failure_rate: float
    severity: str
    timestamp: datetime

class DataQualityFramework:
    """
    Framework simplificado inspirado en Great Expectations
    En producción: usar Great Expectations, Soda o dbt tests
    """
    
    def __init__(self, name: str):
        self.suite_name = name
        self.checks: List[QualityCheck] = []
        self.results: List[QualityResult] = []
    
    def add_check(self, check: QualityCheck):
        """Agrega una verificación a la suite"""
        self.checks.append(check)
        print(f"✓ Check agregado: {check.name}")
    
    def validate(self, df: pd.DataFrame) -> Dict[str, Any]:
        """Ejecuta todas las verificaciones"""
        self.results = []
        print(f"\n🔍 Ejecutando {len(self.checks)} checks...")
        print("-" * 60)
        
        for check in self.checks:
            result = self._run_check(df, check)
            self.results.append(result)
            
            status = "✓ PASS" if result.passed else "✗ FAIL"
            print(f"{status} | {check.name}")
            if not result.passed:
                print(f"       ↳ {result.rows_failed}/{result.rows_evaluated} filas fallaron ({result.failure_rate:.1%})")
        
        return self.get_summary()
    
    def _run_check(self, df: pd.DataFrame, check: QualityCheck) -> QualityResult:
        """Ejecuta una verificación individual"""
        col = check.column
        total_rows = len(df)
        timestamp = datetime.now()
        
        if check.check_type == 'not_null':
            failed = df[col].isna().sum()
            passed = failed == 0
            
        elif check.check_type == 'unique':
            failed = df[col].duplicated().sum()
            passed = failed == 0
            
        elif check.check_type == 'range':
            min_val = check.params.get('min')
            max_val = check.params.get('max')
            failed = (~df[col].between(min_val, max_val)).sum()
            passed = failed == 0
            
        elif check.check_type == 'regex':
            pattern = check.params.get('pattern')
            failed = (~df[col].astype(str).str.match(pattern, na=False)).sum()
            passed = failed == 0
            
        elif check.check_type == 'custom':
            func = check.params.get('function')
            failed = (~func(df[col])).sum()
            passed = failed == 0
        else:
            raise ValueError(f"Check type desconocido: {check.check_type}")
        
        return QualityResult(
            check_name=check.name,
            passed=passed,
            rows_evaluated=total_rows,
            rows_failed=failed,
            failure_rate=failed / total_rows if total_rows > 0 else 0,
            severity=check.severity,
            timestamp=timestamp
        )
    
    def get_summary(self) -> Dict[str, Any]:
        """Genera resumen de resultados"""
        total_checks = len(self.results)
        passed_checks = sum(1 for r in self.results if r.passed)
        failed_checks = total_checks - passed_checks
        
        errors = [r for r in self.results if not r.passed and r.severity == 'error']
        warnings = [r for r in self.results if not r.passed and r.severity == 'warning']
        
        summary = {
            'suite_name': self.suite_name,
            'total_checks': total_checks,
            'passed': passed_checks,
            'failed': failed_checks,
            'success_rate': passed_checks / total_checks if total_checks > 0 else 0,
            'errors': len(errors),
            'warnings': len(warnings),
            'overall_pass': len(errors) == 0
        }
        
        return summary
    
    def print_summary(self):
        """Imprime resumen formateado"""
        summary = self.get_summary()
        
        print("\n" + "=" * 60)
        print(f"📊 RESUMEN DE CALIDAD: {summary['suite_name']}")
        print("=" * 60)
        print(f"Total checks:    {summary['total_checks']}")
        print(f"✓ Passed:        {summary['passed']}")
        print(f"✗ Failed:        {summary['failed']} ({summary['errors']} errors, {summary['warnings']} warnings)")
        print(f"Success rate:    {summary['success_rate']:.1%}")
        print("-" * 60)
        
        if summary['overall_pass']:
            print("🎉 Dataset APROBADO - Cumple todos los checks críticos")
        else:
            print("⚠️  Dataset RECHAZADO - Hay checks críticos fallando")
        print("=" * 60)

# Ejemplo de uso
print("Ejemplo: Framework de Calidad de Datos Moderno\n")

# Crear suite de calidad
suite = DataQualityFramework(name="Clientes Quality Suite")

# Agregar checks
suite.add_check(QualityCheck(
    name="cliente_id no debe ser nulo",
    column="cliente_id",
    check_type="not_null",
    params={},
    severity="error"
))

suite.add_check(QualityCheck(
    name="cliente_id debe ser único",
    column="cliente_id",
    check_type="unique",
    params={},
    severity="error"
))

suite.add_check(QualityCheck(
    name="edad debe estar en rango [0, 120]",
    column="edad",
    check_type="range",
    params={'min': 0, 'max': 120},
    severity="error"
))

suite.add_check(QualityCheck(
    name="email debe tener formato válido",
    column="email",
    check_type="regex",
    params={'pattern': r'^[\w\.-]+@[\w\.-]+\.\w+$'},
    severity="warning"  # warning porque puede ser nulo legítimamente
))

# Validar dataset (reusando df anterior)
suite.validate(df)
suite.print_summary()

## 9. Integración de Calidad en CI/CD

### 9.1 Quality Gates en Pipelines

En arquitecturas modernas, la calidad de datos se valida **antes** de promover datos entre ambientes:

```
┌─────────────┐      ┌─────────────┐      ┌─────────────┐      ┌─────────────┐
│   Bronze    │ ───▶ │   Quality   │ ───▶ │   Silver    │ ───▶ │   Quality   │
│  (Raw Data) │      │    Gate     │      │  (Cleaned)  │      │    Gate     │
└─────────────┘      └─────────────┘      └─────────────┘      └─────────────┘
                            │                                           │
                            ▼                                           ▼
                     ❌ RECHAZA                                   ❌ RECHAZA
                     ✅ APRUEBA                                   ✅ APRUEBA
                                                                         │
                                                                         ▼
                                                                  ┌─────────────┐
                                                                  │    Gold     │
                                                                  │ (Analytics) │
                                                                  └─────────────┘
```

### 9.2 Ejemplo Conceptual: Airflow + Great Expectations

```python
# Pseudocódigo de DAG de Airflow
from airflow import DAG
from airflow.operators.python import PythonOperator
from great_expectations_provider.operators.great_expectations import (
    GreatExpectationsOperator
)

with DAG('etl_with_quality_gates', schedule_interval='@daily') as dag:
    
    # 1. Extraer datos
    extract = PythonOperator(
        task_id='extract_from_source',
        python_callable=extract_data
    )
    
    # 2. Quality gate en Bronze
    validate_bronze = GreatExpectationsOperator(
        task_id='validate_bronze',
        expectation_suite_name='bronze_suite',
        data_asset_name='bronze_clientes',
        fail_task_on_validation_failure=True  # Detiene pipeline si falla
    )
    
    # 3. Transformar a Silver
    transform = PythonOperator(
        task_id='transform_to_silver',
        python_callable=transform_data
    )
    
    # 4. Quality gate en Silver
    validate_silver = GreatExpectationsOperator(
        task_id='validate_silver',
        expectation_suite_name='silver_suite',
        data_asset_name='silver_clientes'
    )
    
    # 5. Cargar a Gold
    load = PythonOperator(
        task_id='load_to_gold',
        python_callable=load_data
    )
    
    extract >> validate_bronze >> transform >> validate_silver >> load
```

### 9.3 Métricas de Calidad como Observabilidad

**Métricas clave a monitorear:**
- **Data Freshness:** ¿Cuándo se actualizó por última vez?
- **Completeness Rate:** % de valores no nulos
- **Duplicate Rate:** % de registros duplicados
- **Anomaly Score:** Desviación de patrones históricos
- **Schema Drift:** Cambios no esperados en estructura

**Dashboard de Calidad (concepto):**
```
┌──────────────────────────────────────────────────────────────┐
│  Quality Dashboard - Clientes                        🟢 96%  │
├──────────────────────────────────────────────────────────────┤
│                                                              │
│  📊 Completeness:  98% ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓░  [Target: 95%]  │
│  🔍 Uniqueness:    100% ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓ [Target: 100%] │
│  ⏰ Freshness:     2h ago 🟢                [Target: <4h]    │
│  ⚠️  Anomalies:    3 detected 🟡            [Threshold: 5]  │
│  📋 Schema:        Stable ✓                 [No changes]     │
│                                                              │
│  Last validation: 2025-11-22 14:30 UTC                      │
└──────────────────────────────────────────────────────────────┘
```

### 9.4 Alertas Automatizadas

**Configuración de alertas:**
```yaml
# alerts.yml
alerts:
  - name: "Completeness drop"
    condition: completeness < 90%
    severity: critical
    channel: slack:#data-quality
    
  - name: "High duplicates"
    condition: duplicate_rate > 5%
    severity: warning
    channel: email:data-team@company.com
    
  - name: "Data freshness"
    condition: last_update_hours > 4
    severity: critical
    channel: pagerduty
```

## 18. Ejercicio Transversal de Calidad de Datos (Nivel Junior)

Este ejercicio aplica conceptos de perfilado, reglas de calidad y documentación usando el dataset `sales_orders.csv` localizado en `../04-recursos/datasets/` (ruta absoluta: `04-recursos/datasets/sales_orders.csv`). Complementa la plantilla general en `../04-recursos/plantilla_ejercicio_transversal.md`.

### Objetivo
Detectar, cuantificar y remediar anomalías clave para preparar la transición Bronze → Silver con trazabilidad mínima.

### Anomalías Esperadas
- `order_id` duplicado.
- `unit_price` negativo.
- `total_amount` inconsistente (≠ `quantity * unit_price`).
- Fechas fuera de rango (futuras o muy antiguas).
- Estados incoherentes (ej. CANCELLED con monto positivo).

### Tareas
1. Cargar el dataset y mostrar shape + 5 primeras filas.
2. Generar perfil simple (conteo de nulos, tipos, cardinalidad básica).
3. Calcular tabla de anomalías: tipo, conteo, porcentaje sobre total.
4. Definir 5 reglas de calidad (data contract preliminar) con: expresión/pseudocódigo, severidad, métrica esperada (% válido), umbral.
5. Diseñar estrategia de remediación (bullet list) para cada anomalía (drop, corregir, flag).
6. Crear mini flujo Bronze → Silver (lista numerada) indicando formato, particionado y justificación.
7. Implementar función `run_quality_checks(df)` que devuelva dict con métricas de cada regla.
8. Redactar mini ADR (máx. 8 líneas) justificando elección de formato en Silver (Parquet + particionado por `order_date` vs CSV plano).

### Rúbrica
| Aspecto | Peso |
|---------|------|
| Cuantificación de anomalías | 25% |
| Reglas claras y medibles | 25% |
| Remediación y flujo Bronze→Silver | 25% |
| Implementación función checks | 15% |
| ADR conciso | 10% |

### Siguientes Pasos (Opcional)
- Integrar Great Expectations suite mínima para 2 reglas críticas.
- Registrar contrato en repositorio (versión v0.1).
- Añadir métrica de cobertura de reglas en dashboard interno.

### Referencias
- Plantilla transversal: `../04-recursos/plantilla_ejercicio_transversal.md`
- Dataset: `../04-recursos/datasets/sales_orders.csv`

> Este bloque prepara la elevación del mismo dataset que se verá con fines de streaming y features en el nivel Mid.


In [ ]:
# Código base para el ejercicio transversal de calidad (Junior)
import pandas as pd
from pathlib import Path

DATA_PATH_REL = Path("../04-recursos/datasets/sales_orders.csv")
DATA_PATH_ROOT = Path("04-recursos/datasets/sales_orders.csv")

# Intentamos carga por ruta relativa primero, fallback a raíz si se ejecuta desde carpeta root.
if DATA_PATH_REL.exists():
    csv_path = DATA_PATH_REL
elif DATA_PATH_ROOT.exists():
    csv_path = DATA_PATH_ROOT
else:
    raise FileNotFoundError("No se encontró sales_orders.csv en rutas esperadas")

df = pd.read_csv(csv_path)
print("Shape:", df.shape)
print(df.head())

# Perfil simple
profile = {
    'n_rows': len(df),
    'n_columns': len(df.columns),
    'null_counts': df.isna().sum().to_dict(),
    'dtypes': df.dtypes.astype(str).to_dict(),
    'cardinality': {c: df[c].nunique() for c in df.columns}
}
print("Perfil básico:")
for k,v in profile.items():
    print(k,":", v if k != 'null_counts' else '...')

# Detección de anomalías
anomalies = {}
# 1. order_id duplicado
anomalies['duplicate_order_id'] = df['order_id'].duplicated().sum()
# 2. unit_price negativo
anomalies['negative_unit_price'] = (df['unit_price'] < 0).sum()
# 3. total_amount inconsistente
anomalies['inconsistent_total_amount'] = (df['total_amount'] != df['quantity'] * df['unit_price']).sum()
# 4. fechas fuera de rango (definimos rango arbitrario 2018-01-01 a 2025-12-31)
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
mask_date = (df['order_date'] < pd.Timestamp('2018-01-01')) | (df['order_date'] > pd.Timestamp('2025-12-31'))
anomalies['out_of_range_order_date'] = mask_date.sum()
# 5. estado incoherente (CANCELLED con total_amount > 0)
anomalies['cancelled_positive_amount'] = ((df['status'] == 'CANCELLED') & (df['total_amount'] > 0)).sum()

print("\nTabla anomalías:")
print("tipo, conteo, porcentaje")
for k,v in anomalies.items():
    pct = (v / len(df)) * 100 if len(df) else 0
    print(f"{k}, {v}, {pct:.2f}%")

# Reglas preliminares (ejemplo estructurado)
quality_rules = [
    {"name": "unique_order_id", "expr": "no duplicados", "severity": "Alta", "metric": "duplicados_pct", "threshold": "< 0.5%"},
    {"name": "non_negative_price", "expr": "unit_price >= 0", "severity": "Alta", "metric": "precios_validos_pct", "threshold": "> 99%"},
    {"name": "consistent_total", "expr": "total_amount == quantity * unit_price", "severity": "Media", "metric": "totales_consistentes_pct", "threshold": "> 98%"},
    {"name": "valid_date_range", "expr": "2018-01-01 <= order_date <= 2025-12-31", "severity": "Media", "metric": "fechas_validas_pct", "threshold": "> 99%"},
    {"name": "cancelled_amount_zero", "expr": "status != CANCELLED or total_amount == 0", "severity": "Baja", "metric": "cancelled_correcto_pct", "threshold": "> 97%"}
]

# Función de evaluación de reglas

def run_quality_checks(df: pd.DataFrame) -> dict:
    total = len(df)
    results = {}
    # Calculamos métricas alineadas a reglas
    results['duplicados_pct'] = (df['order_id'].duplicated().sum() / total) * 100 if total else 0
    results['precios_validos_pct'] = ((df['unit_price'] >= 0).sum() / total) * 100 if total else 0
    results['totales_consistentes_pct'] = ((df['total_amount'] == df['quantity'] * df['unit_price']).sum() / total) * 100 if total else 0
    results['fechas_validas_pct'] = (~mask_date).sum() / total * 100 if total else 0
    results['cancelled_correcto_pct'] = (((df['status'] != 'CANCELLED') | (df['total_amount'] == 0)).sum() / total) * 100 if total else 0
    return results

metrics = run_quality_checks(df)
print("\nMétricas reglas calidad:")
for k,v in metrics.items():
    print(f"{k}: {v:.2f}%")

# Placeholder ADR (rellenar manualmente por el estudiante)
adr_template = """ADR: Formato de Almacenamiento Silver\nContexto: Necesitamos formato columnar eficiente para analítica y compresión.\nDecisión: Usar Parquet particionado por order_date (AÑO-MES).\nAlternativas: CSV (sin compresión/lectura lenta), Avro (orientado a fila), Delta (overhead adicional si aún no necesitamos transacciones).\nConsecuencias Positivas: Menor costo almacenamiento, lectura selectiva columnas.\nConsecuencias Negativas: Complejidad particionado y manejo de esquemas.\nRiesgos: Evolución de esquema (añadir columnas).\nMitigación: Definir política de schema evolution y pruebas en entorno staging.\n"""
print("\nADR template:\n", adr_template)
